# ReST - imbalanced-target robustness test
Test on **CIFAR-10** with two target-sampling schemes:
* **balanced**   - 200 uniform-random samples (baseline).
* **imbalanced** - **150 samples of class 5**, **5 samples of each other class** (195 total).

In [1]:
!pip install -q datasets pillow scipy


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, logging, warnings
os.environ["HF_HUB_OFFLINE"]              = "1"   
os.environ["HF_DATASETS_OFFLINE"]         = "1"
os.environ["TRANSFORMERS_VERBOSITY"]      = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]= "1"
os.environ["TOKENIZERS_PARALLELISM"]      = "false"

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)
for _n in ("datasets", "huggingface_hub", "transformers", "urllib3", "filelock", "fsspec"):
    logging.getLogger(_n).setLevel(logging.ERROR)
try:
    import datasets
    datasets.logging.set_verbosity_error()
    try: datasets.disable_progress_bars()
    except Exception: pass
except Exception:
    pass
print("warnings silenced; Hugging Face in offline/cache-only mode")

warnings silenced; Hugging Face in offline/cache-only mode


In [3]:
import os, json, warnings
from collections import defaultdict
from typing import Dict, List, Tuple, Any, Optional, Set

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as T
import torchvision.models as models
import torchvision.datasets as tvds

warnings.filterwarnings("ignore")

def set_seed(seed: int = 42):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def feature_downproj(x: torch.Tensor, max_feat: Optional[int] = 4096) -> torch.Tensor:
    if max_feat is None:
        return x
    B, D = x.shape
    if D <= max_feat:
        return x
    gen = torch.Generator(device=x.device).manual_seed(12345)
    P = torch.randn(D, max_feat, generator=gen, device=x.device) / np.sqrt(max_feat)
    return x @ P

def compute_stable_rank_from_svals(s, eps=1e-8):
    if s.numel() == 0: return 0.0
    return (torch.sum(s**2) / (torch.max(s)**2 + eps)).item()

def compute_ranks(matrix):
    if matrix.numel() == 0: return 0.0
    s = torch.linalg.svdvals(matrix)
    return compute_stable_rank_from_svals(s)

def _find_module_name(model, target):
    for name, mod in model.named_modules():
        if mod is target: return name
    return None

def resolve_final_classifier_names(model) -> Set[str]:
    names = set()
    if hasattr(model, "fc") and isinstance(model.fc, nn.Linear):
        n = _find_module_name(model, model.fc); names.add(n) if n else None
    if hasattr(model, "classifier"):
        clf = model.classifier
        if isinstance(clf, nn.Linear):
            n = _find_module_name(model, clf); names.add(n) if n else None
        elif isinstance(clf, nn.Sequential) and len(clf) > 0 and isinstance(clf[-1], nn.Linear):
            n = _find_module_name(model, clf[-1]); names.add(n) if n else None
    return names

def resolve_final_classifier_module(model) -> Optional[nn.Linear]:
    if hasattr(model, "fc") and isinstance(model.fc, nn.Linear): return model.fc
    if hasattr(model, "classifier"):
        clf = model.classifier
        if isinstance(clf, nn.Linear): return clf
        if isinstance(clf, nn.Sequential) and len(clf) > 0 and isinstance(clf[-1], nn.Linear): return clf[-1]
    return None

def find_last_weighted_module(model, exclude_names: Set[str]):
    last_mod, last_name = None, None
    for name, m in model.named_modules():
        if name in exclude_names: continue
        if isinstance(m, (nn.Linear, nn.Conv1d, nn.Conv2d, nn.Conv3d)) and isinstance(getattr(m, "weight", None), torch.Tensor):
            last_mod, last_name = m, name
    return last_mod, last_name

def _weight_matrix_from_module(mod):
    W = getattr(mod, "weight", None)
    if not isinstance(W, torch.Tensor): return None, None, None
    orig_shape = list(W.shape)
    if isinstance(mod, nn.Linear):
        mat = W.detach().float()
    elif isinstance(mod, (nn.Conv1d, nn.Conv2d, nn.Conv3d)):
        mat = W.detach().float().view(W.shape[0], -1)
    else:
        return None, orig_shape, None
    return mat.cpu(), orig_shape, list(mat.shape)

def compute_weight_ranks(mod):
    if mod is None:
        return {"stable_rank": None, "shape": None, "matrix_shape": None}
    mat, orig_shape, mat_shape = _weight_matrix_from_module(mod)
    if mat is None:
        return {"stable_rank": None, "shape": orig_shape, "matrix_shape": mat_shape}
    sr = compute_ranks(mat)
    return {"stable_rank": sr, "shape": orig_shape, "matrix_shape": mat_shape}

# ---- activation extractor --------------------------------------------------
class ActivationExtractor:
    def __init__(self, model, device="cuda", max_feats_per_layer=4096, final_exclude_names=None, classifier_module=None):
        self.model = model.to(device); self.device = device
        self.activations = {}; self.hooks = []
        self.max_feats_per_layer = max_feats_per_layer
        self.final_exclude_names = final_exclude_names or set()
        self.penult_name = None
        self.classifier_module = classifier_module
        self.classifier_inputs = []; self.classifier_outputs = []

    def _keep(self, name, module):
        if name in self.final_exclude_names: return False
        if isinstance(module, (nn.Sequential, nn.ModuleList, nn.Identity)): return False
        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.Dropout, nn.Dropout2d, nn.Dropout3d)): return False
        return True

    def register_hooks(self):
        def hook_fn(name):
            def _hook(_m, _i, output):
                x = output[0] if isinstance(output, (list, tuple)) and output else output
                if not isinstance(x, torch.Tensor): return
                self.penult_name = name
                flat = feature_downproj(x.view(x.shape[0], -1), self.max_feats_per_layer)
                self.activations[name] = flat.detach()
            return _hook
        for name, module in self.model.named_modules():
            if self._keep(name, module):
                self.hooks.append(module.register_forward_hook(hook_fn(name)))
        if self.classifier_module is not None:
            def pre_hook(_m, x):
                x0 = x[0]
                if isinstance(x0, torch.Tensor):
                    flat = feature_downproj(x0.view(x0.shape[0], -1), self.max_feats_per_layer)
                    self.classifier_inputs.append(flat.detach().cpu())
            def classifier_hook(_m, _i, output):
                out = output[0] if isinstance(output, (list, tuple)) and output else output
                if isinstance(out, torch.Tensor):
                    self.classifier_outputs.append(out.view(out.shape[0], -1).detach().cpu())
            self.hooks.append(self.classifier_module.register_forward_pre_hook(pre_hook))
            self.hooks.append(self.classifier_module.register_forward_hook(classifier_hook))

    def remove_hooks(self):
        for h in self.hooks: h.remove()
        self.hooks.clear()

    @torch.no_grad()
    def extract_activations(self, dataloader, max_samples=None):
        self.model.eval()
        buf = defaultdict(list); seen = 0
        self.classifier_inputs = []; self.classifier_outputs = []
        for images, _ in dataloader:
            if max_samples is not None and seen >= max_samples: break
            self.activations.clear()
            images = images.to(self.device, non_blocking=True)
            _ = self.model(images)
            for lname, act in self.activations.items():
                if act is not None: buf[lname].append(act.cpu())
            seen += images.size(0)
        return {k: torch.cat(v, 0) for k, v in buf.items() if v}

def load_model(model_name, pretrained=True):
    name = model_name.lower()
    table = {
        "googlenet":   lambda: models.googlenet(weights=models.GoogLeNet_Weights.DEFAULT if pretrained else None, aux_logits=True),
        "mobilenet_v2":lambda: models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None),
        "mnasnet1_0":  lambda: models.mnasnet1_0(weights=models.MNASNet1_0_Weights.DEFAULT if pretrained else None),
        "densenet121": lambda: models.densenet121(weights=models.DenseNet121_Weights.DEFAULT if pretrained else None),
        "densenet169": lambda: models.densenet169(weights=models.DenseNet169_Weights.DEFAULT if pretrained else None),
        "densenet201": lambda: models.densenet201(weights=models.DenseNet201_Weights.DEFAULT if pretrained else None),
        "resnet34":    lambda: models.resnet34(weights=models.ResNet34_Weights.DEFAULT if pretrained else None),
        "resnet50":    lambda: models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None),
        "resnet101":   lambda: models.resnet101(weights=models.ResNet101_Weights.DEFAULT if pretrained else None),
        "resnet152":   lambda: models.resnet152(weights=models.ResNet152_Weights.DEFAULT if pretrained else None),
    }
    if name not in table:
        raise ValueError(f"Unknown: {model_name}")
    m = table[name]()
    if name == "googlenet":
        m.aux_logits = False
    return m

def default_transform(model_name):
    return T.Compose([T.Resize((224, 224)), T.ToTensor(),
                      T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

class HFDatasetTorchWrapper(torch.utils.data.Dataset):
    def __init__(self, hf_ds, transform): self.ds, self.transform = hf_ds, transform
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        ex = self.ds[int(idx)]
        return self.transform(ex["image"]), int(ex["label"])

def _load_hf_mini_imagenet(split, transform):
    from datasets import load_dataset
    # No trust_remote_code (rejected by newer `datasets`); offline env vars force cache use.
    ds = load_dataset("timm/mini-imagenet", split=("train" if split == "train" else "validation"))
    return HFDatasetTorchWrapper(ds, transform)

def get_dataset(dataset_name, transform, split="train"):
    name = dataset_name.lower()
    if name == "cifar10":  return tvds.CIFAR10(root="./data", train=(split == "train"), transform=transform, download=True)
    if name == "cifar100": return tvds.CIFAR100(root="./data", train=(split == "train"), transform=transform, download=True)
    if name in {"mini_imagenet"}: return _load_hf_mini_imagenet(split, transform)
    raise ValueError(f"Unsupported dataset: {dataset_name}")

def create_random_subset(dataset, num_samples=200, seed=1234):
    n = len(dataset)
    k = min(num_samples, n)
    if seed is None:
        perm = torch.randperm(n)
    else:
        g = torch.Generator().manual_seed(seed)
        perm = torch.randperm(n, generator=g)
    return Subset(dataset, perm[:k].tolist())

def create_imbalanced_subset(dataset, class_counts, default_count=5, seed=1234):
    """Class-imbalanced subset: class_counts maps class_id -> #samples; every other class
    gets default_count. Reproducible via seed."""
    targets = getattr(dataset, "targets", None)
    if targets is None:
        targets = [dataset[i][1] for i in range(len(dataset))]
    targets = torch.as_tensor(targets).view(-1)
    g = torch.Generator().manual_seed(seed)
    idxs = []
    for c in torch.unique(targets).tolist():
        want = int(class_counts.get(c, default_count))
        c_idx = torch.where(targets == c)[0]
        sel = c_idx[torch.randperm(len(c_idx), generator=g)[:want]].tolist() if len(c_idx) > want else c_idx.tolist()
        idxs.extend(sel)
    return Subset(dataset, idxs)

def calculate_transferability_scores(model_name, dataset_name, num_samples=200, sample_seed=1234,
                                     device="cuda", batch_size=32, max_feats_per_layer=4096, max_samples=None, imbalance=None, imbalance_default=5):
    transform = default_transform(model_name)
    model = load_model(model_name, pretrained=True)
    final_exclude_names = resolve_final_classifier_names(model)
    clf_mod = resolve_final_classifier_module(model)

    _ds = get_dataset(dataset_name, transform, "train")
    if imbalance is not None:
        subset = create_imbalanced_subset(_ds, imbalance, imbalance_default, seed=sample_seed)
    else:
        subset = create_random_subset(_ds, num_samples=num_samples, seed=sample_seed)
    dataloader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=(device == "cuda"))

    ext = ActivationExtractor(model, device, max_feats_per_layer, final_exclude_names, clf_mod)
    ext.register_hooks()
    try:
        activations = ext.extract_activations(dataloader, max_samples=max_samples)
        penult_name = ext.penult_name
    finally:
        ext.remove_hooks()

    if not activations and not ext.classifier_inputs and not ext.classifier_outputs:
        return None

    penult, penult_source = None, "last_non_excluded"
    if ext.classifier_inputs:
        penult = torch.cat(ext.classifier_inputs, 0); penult_source = "classifier_pre_hook"
    elif penult_name and penult_name in activations:
        penult = activations[penult_name]
    elif activations:
        penult = activations[next(reversed(activations))]

    pen_sr = 0.0
    if penult is not None:
        pen_sr = compute_ranks(penult.float().T)

    clf_sr = 0.0; clf_tensor = None; clf_source = "none"
    if ext.classifier_outputs:
        clf_tensor = torch.cat(ext.classifier_outputs, 0); clf_source = "classifier_hook"
        clf_sr = compute_ranks(clf_tensor.float().T)

    pen_w_mod, pen_w_name = find_last_weighted_module(model, final_exclude_names)
    return {
        "penultimate_layer": [pen_sr],
        "classifier_layer":  [clf_sr],
        "penultimate_layer_source": penult_source,
        "classifier_layer_source": clf_source,
        "activation_space_shape": {
            "penultimate_layer": list(penult.shape) if penult is not None else None,
            "classifier_layer":  list(clf_tensor.shape) if clf_tensor is not None else None,
        },
        "weight_penultimate": compute_weight_ranks(pen_w_mod),
        "weight_classifier":  compute_weight_ranks(clf_mod),
    }

In [4]:
# =====================  CONFIG  =====================
import os
SEED=123; GAMMA=0.2; NUM_SAMPLES=200; SAMPLE_SEED=1234; BATCH_SIZE=32; MAX_FEATS=4096
MODEL_HUB=["mobilenet_v2","mnasnet1_0","densenet121","densenet169","densenet201",
           "resnet34","resnet50","resnet101","resnet152","googlenet"]
SOURCE_DATASET="mini_imagenet"

# imbalance spec: CIFAR-10 class 5 -> 150 samples, every other class -> 5
IMBALANCE={5:150}; IMBALANCE_DEFAULT=5
VARIANTS={"cifar10_balanced": None, "cifar10_imbalanced": IMBALANCE}   # None = random 200

CIFAR10_ACC={"resnet34":96.12,"resnet50":96.28,"resnet101":97.39,"resnet152":97.53,
             "densenet121":96.45,"densenet169":96.77,"densenet201":97.02,
             "mnasnet1_0":92.59,"mobilenet_v2":94.74,"googlenet":95.54}

OUT_DIR="rest_json_imbalanced"; os.makedirs(OUT_DIR,exist_ok=True)
device="cuda" if __import__("torch").cuda.is_available() else "cpu"
print("device:",device)

device: cuda


In [5]:
from collections import Counter
_tf=default_transform("resnet50")
_ds=get_dataset("cifar10",_tf,"train")
_imb=create_imbalanced_subset(_ds, IMBALANCE, IMBALANCE_DEFAULT, seed=SAMPLE_SEED)
_dist=Counter(int(_ds.targets[i]) for i in _imb.indices)
print("imbalanced subset size:", len(_imb.indices))
print("per-class counts:", dict(sorted(_dist.items())))

Files already downloaded and verified
imbalanced subset size: 195
per-class counts: {0: 5, 1: 5, 2: 5, 3: 5, 4: 5, 5: 150, 6: 5, 7: 5, 8: 5, 9: 5}


In [6]:
# =====================  STEP 1-2: load source (JSON) ; extract balanced/imbalanced CIFAR-10  =====================
import json
set_seed(SEED)

def _display_record(model_name, rec):
    print(f"  {model_name:<13} pen_sr={rec['penultimate_layer'][0]:.3f} "
          f"clf_sr={rec['classifier_layer'][0]:.3f} "
          f"w_pen={rec['weight_penultimate']['stable_rank']:.3f} "
          f"w_clf={rec['weight_classifier']['stable_rank']:.3f}")

all_json = {}

SOURCE_JSON = os.path.join(OUT_DIR, f"{SOURCE_DATASET}_transferability_scores.json")
with open(SOURCE_JSON, "r") as f:
    all_json[SOURCE_DATASET] = json.load(f)
print(f"=== {SOURCE_DATASET} ===")
for model_name, rec in all_json[SOURCE_DATASET].items():
    _display_record(model_name, rec)

# ---- TARGETS: CIFAR-10 balanced + imbalanced (extract, or load cached) ----
RUNS = [
    ("cifar10_imbalanced", "cifar10", IMBALANCE, CIFAR10_ACC),
    ("cifar10_balanced",   "cifar10", None,      CIFAR10_ACC),
]
for store_name, real_dataset, imbalance, gt in RUNS:
    print(f"\n=== {store_name} ===")
    json_path = os.path.join(OUT_DIR, f"{store_name}_transferability_scores.json")

    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            ds_results = json.load(f)
        for model_name, rec in ds_results.items():
            _display_record(model_name, rec)
        all_json[store_name] = ds_results
        continue

    ds_results = {}
    for model_name in MODEL_HUB:
        try:
            rec = calculate_transferability_scores(
                model_name, real_dataset,
                num_samples=NUM_SAMPLES, device=device,
                batch_size=BATCH_SIZE, max_feats_per_layer=MAX_FEATS,
                imbalance=imbalance, imbalance_default=IMBALANCE_DEFAULT,
            )
        except Exception as e:
            print(f"  [skip] {model_name}: {e}")
            continue
        if rec is None:
            print(f"  [skip] {model_name}: no activations")
            continue
        rec["finetune_accuracy"] = gt.get(model_name) if gt else None
        ds_results[model_name] = rec
        _display_record(model_name, rec)

    all_json[store_name] = ds_results
    with open(json_path, "w") as f:
        json.dump(ds_results, f, indent=2)

print("\nSaved / loaded JSONs from", OUT_DIR)

=== mini_imagenet ===
  mobilenet_v2  pen_sr=1.874 clf_sr=1.963 w_pen=40.377 w_clf=64.369
  mnasnet1_0    pen_sr=1.668 clf_sr=1.683 w_pen=102.206 w_clf=37.585
  densenet121   pen_sr=1.448 clf_sr=1.669 w_pen=16.097 w_clf=40.178
  densenet169   pen_sr=1.483 clf_sr=1.808 w_pen=12.439 w_clf=46.056
  densenet201   pen_sr=1.518 clf_sr=1.624 w_pen=13.949 w_clf=47.066
  resnet34      pen_sr=1.301 clf_sr=1.611 w_pen=100.397 w_clf=36.082
  resnet50      pen_sr=1.438 clf_sr=1.491 w_pen=5.591 w_clf=97.067
  resnet101     pen_sr=1.336 clf_sr=1.354 w_pen=5.787 w_clf=97.922
  resnet152     pen_sr=1.379 clf_sr=1.358 w_pen=5.797 w_clf=101.002
  googlenet     pen_sr=1.417 clf_sr=2.006 w_pen=252.577 w_clf=60.079

=== cifar10_imbalanced ===
Files already downloaded and verified
  mobilenet_v2  pen_sr=1.881 clf_sr=1.754 w_pen=40.377 w_clf=64.369
Files already downloaded and verified
  mnasnet1_0    pen_sr=1.720 clf_sr=1.708 w_pen=102.206 w_clf=37.585
Files already downloaded and verified
  densenet121   pe

In [7]:
# =====================  STEP 3: build the four ReST elements (pen_act = target - source)  =====================
import pandas as pd
src = all_json[SOURCE_DATASET]
TARGETS = list(VARIANTS)   # cifar10_balanced, cifar10_imbalanced
rows=[]
for dataset_name in TARGETS:
    for model_name, d in all_json[dataset_name].items():
        if model_name not in src: continue
        s=src[model_name]
        rows.append({
            "target dataset":dataset_name,"pre-trained model":model_name,
            "pen_act": d["penultimate_layer"][0]/min(d["activation_space_shape"]["matrix_shape"])-s["penultimate_layer"][0],
            "clf_act": d["classifier_layer"][0]-s["classifier_layer"][0],
            "pen_before_weight": d["weight_penultimate"]["stable_rank"]/min(d["weight_penultimate"]["matrix_shape"]),
            "clf_before_weight": d["weight_classifier"]["stable_rank"]/min(d["weight_classifier"]["matrix_shape"]),
            "fine-tune accuracy": d["finetune_accuracy"],
        })
df=pd.DataFrame(rows); df

,target dataset,pre-trained model,pen_act,clf_act,pen_before_weight,clf_before_weight,fine-tune accuracy
0,cifar10_balanced,mobilenet_v2,0.091611,-0.034813,0.126178,0.064369,94.74
1,cifar10_balanced,mnasnet1_0,0.093365,0.324696,0.319393,0.037585,92.59
2,cifar10_balanced,densenet121,0.143782,0.717783,0.503019,0.040178,96.45
3,cifar10_balanced,densenet169,0.275525,0.509553,0.388716,0.046056,96.77
4,cifar10_balanced,densenet201,0.343574,0.855108,0.435900,0.047066,97.02
5,cifar10_balanced,resnet34,0.183541,0.469180,0.196088,0.070474,96.12
6,cifar10_balanced,resnet50,0.920955,0.560087,0.010920,0.097067,96.28
7,cifar10_balanced,resnet101,1.697882,2.011922,0.011304,0.097922,97.39
8,cifar10_balanced,resnet152,1.913614,2.192365,0.011322,0.101002,97.53
9,cifar10_balanced,googlenet,0.092393,0.145397,0.252577,0.060079,95.54


In [8]:
# =====================  STEP 4: ReST score (z-score per variant)  =====================
from scipy.stats import zscore
FEATURES=["pen_act","clf_act","pen_before_weight","clf_before_weight"]
df=df.copy()
for _,sub in df.groupby("target dataset"):
    df.loc[sub.index,FEATURES]=sub[FEATURES].apply(zscore)
G=df["pen_before_weight"]+df["clf_before_weight"]
L=df["pen_act"]+df["clf_act"]
df["ReST"]=(1-GAMMA)*G+GAMMA*L
df[["target dataset","pre-trained model","ReST","fine-tune accuracy"]]

,target dataset,pre-trained model,ReST,fine-tune accuracy
0,cifar10_balanced,mobilenet_v2,-0.891701,94.74
1,cifar10_balanced,mnasnet1_0,-0.819886,92.59
2,cifar10_balanced,densenet121,0.234689,96.45
3,cifar10_balanced,densenet169,-0.106371,96.77
4,cifar10_balanced,densenet201,0.262109,97.02
5,cifar10_balanced,resnet34,-0.193451,96.12
6,cifar10_balanced,resnet50,0.116307,96.28
7,cifar10_balanced,resnet101,0.792824,97.39
8,cifar10_balanced,resnet152,1.014354,97.53
9,cifar10_balanced,googlenet,-0.408873,95.54


In [12]:
# =====================  STEP 5: weighted Kendall vs ground-truth (balanced vs imbalanced)  =====================
import numpy as np
from scipy.stats import weightedtau
print(f"CIFAR-10  ReST weighted-Kendall (gamma={GAMMA})")
print("-"*46)
for dataset_name in TARGETS:
    sub=df[df["target dataset"]==dataset_name]
    acc=np.array(sub["fine-tune accuracy"].to_numpy(float),copy=True)
    rest=np.array(sub["ReST"].to_numpy(float),copy=True)
    tau=weightedtau(acc,rest).correlation
    print(f"  {dataset_name:<20} tau = {tau:.4f}")
print("-"*46)
print("balanced = 200 random | imbalanced = 150x class5 + 5x each other class")

CIFAR-10  ReST weighted-Kendall (gamma=0.2)
----------------------------------------------
  cifar10_balanced     tau = 0.9214
  cifar10_imbalanced   tau = 0.9214
----------------------------------------------
balanced = 200 random | imbalanced = 150x class5 + 5x each other class
